In [ ]:
import pandas as pd
import psutil
from pathlib import Path

# where everything lives and where the raw 44GB file sits
OUTPUT_DIR = Path(r"C:\mit_supercloud_raw")
RAW_FILE = r"C:\mit_supercloud_raw\nvidia_smi.csv"
CHUNKSIZE = 250_000  # read the file 250k rows at a time so it doesn't blow up my RAM
LOOKUP_FILE = OUTPUT_DIR / "job_start_lookup.csv"  # this file will remember each job's true start time

GROUP_KEYS = ["id_job", "Node", "gpu_index"]  # these 3 columns together uniquely identify one job on one GPU

def ram_usage():
    # just a little helper so I can see how much memory I'm using while this runs
    mem = psutil.virtual_memory()
    return f"{mem.used/1e9:.1f}/{mem.total/1e9:.1f} GB ({mem.percent}%)"

def clean_keys(chunk):
    # forcing these columns into proper number types, since raw CSVs sometimes read things as text
    chunk["id_job"] = pd.to_numeric(chunk["id_job"], errors="coerce")
    chunk["gpu_index"] = pd.to_numeric(chunk["gpu_index"], errors="coerce")
    chunk["timestamp"] = pd.to_numeric(chunk["timestamp"], errors="coerce")
    chunk["Node"] = chunk["Node"].astype(str).str.strip()  # clean up any stray spaces in node names
    chunk.dropna(subset=["id_job", "gpu_index", "timestamp"], inplace=True)  # toss any row that failed to convert
    chunk["id_job"] = chunk["id_job"].astype("int64")  # int64 so big job IDs don't overflow
    chunk["gpu_index"] = chunk["gpu_index"].astype("int64")
    return chunk

# =====================================================================
# Pass 1: figure out exactly when each job actually started
# (only needs to happen ONCE, ever — every later pass reuses this)
# =====================================================================
if LOOKUP_FILE.exists():
    # already did this before, no need to redo it — just load what I saved
    print(f"✅ Found existing lookup table at {LOOKUP_FILE} — skipping Pass 1 entirely.")
    job_start_lookup = pd.read_csv(LOOKUP_FILE)
else:
    print("=" * 60)
    print("PASS 1: scanning whole file for true job start times (one-time only)...")
    print(f"   Initial RAM : {ram_usage()}")
    print("=" * 60)

    pass1_cols = GROUP_KEYS + ["timestamp"]  # only pulling 4 columns here, keeps this pass light and fast
    start_ts_chunks = []  # will collect the "earliest seen so far" from every chunk

    for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=pass1_cols, chunksize=CHUNKSIZE, low_memory=False)):
        chunk = clean_keys(chunk)
        # for this chunk, find the smallest timestamp per job — just this chunk's view, not the whole file yet
        chunk_mins = chunk.groupby(GROUP_KEYS, as_index=False)["timestamp"].min()
        chunk_mins = chunk_mins.rename(columns={"timestamp": "chunk_min_ts"})
        start_ts_chunks.append(chunk_mins)
        del chunk  # free up memory, no need to keep the raw chunk once I've pulled what I need
        if (i + 1) % 100 == 0:
            print(f"  ✅ Pass 1 - Chunk {i+1:>4} | RAM: {ram_usage()}")

    # now stack every chunk's "earliest seen" together, and take the true minimum across ALL of them
    # this is the fix for the earlier bug — a job split across two chunks now gets its real true start time
    all_mins = pd.concat(start_ts_chunks, ignore_index=True)
    job_start_lookup = all_mins.groupby(GROUP_KEYS, as_index=False)["chunk_min_ts"].min()
    job_start_lookup = job_start_lookup.rename(columns={"chunk_min_ts": "job_start_ts"})

    job_start_lookup.to_csv(LOOKUP_FILE, index=False)  # save it so I never have to redo this pass again
    print(f"✅ PASS 1 DONE. Saved lookup table → {LOOKUP_FILE}")
    del start_ts_chunks, all_mins  # cleanup, don't need these anymore

print(f"   Loaded {len(job_start_lookup):,} job-node-gpu start times.\n")

# =====================================================================
# PASS 2 — build the training dataset for the 10-second window
# =====================================================================
EARLY_WINDOW_SEC = 10  # this is the only thing that changes between passes — how many seconds count as "early"
OUTPUT_FILENAME = f"model_a_early_{EARLY_WINDOW_SEC}sec_vs_full_v4.csv"

FEATURES = [
    "power_draw_W", "utilization_gpu_pct", "utilization_memory_pct",
    "memory_used_MiB", "memory_free_MiB", "pcie_link_width_current",
    "temperature_gpu", "temperature_memory",
]

print("=" * 60)
print(f"PASS 2: extracting features for {EARLY_WINDOW_SEC}-second window...")
print("=" * 60)

early_chunk_aggs = []  # will hold the "first N seconds" summaries, per chunk
full_chunk_aggs = []   # will hold the "whole job" summaries, per chunk
pass2_cols = GROUP_KEYS + FEATURES + ["timestamp"]  # this time I need all 8 feature columns too

for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=pass2_cols, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = clean_keys(chunk)
    for col in FEATURES:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")  # make sure every feature column is actually numeric

    # attach each row's TRUE job start time from the lookup table I built in Pass 1
    chunk = chunk.merge(job_start_lookup, on=GROUP_KEYS, how="left")
    chunk["seconds_in"] = chunk["timestamp"] - chunk["job_start_ts"]  # how far into the job is this specific row

    # ---- whole-job stats: sum + count (for the average) + max (for the peak) ----
    agg_dict_full = {col: ["sum", "count", "max"] for col in FEATURES}
    full_agg = chunk.groupby(GROUP_KEYS).agg(agg_dict_full)
    full_agg.columns = ["_".join(c) for c in full_agg.columns]  # flatten the multi-level column names pandas creates
    full_chunk_aggs.append(full_agg.reset_index())

    # ---- early-window stats: just sum + count, since I only need the average here ----
    agg_dict_early = {col: ["sum", "count"] for col in FEATURES}
    # only keep rows that fall within the first N seconds of THIS job (and make sure seconds_in isn't negative/broken)
    early_chunk = chunk[(chunk["seconds_in"] >= 0) & (chunk["seconds_in"] <= EARLY_WINDOW_SEC)]
    if len(early_chunk):  # some chunks might not have any rows this early, so check before grouping
        early_agg = early_chunk.groupby(GROUP_KEYS).agg(agg_dict_early)
        early_agg.columns = ["_".join(c) for c in early_agg.columns]
        early_chunk_aggs.append(early_agg.reset_index())

    del chunk  # free memory before moving to the next chunk
    if (i + 1) % 100 == 0:
        print(f"  ✅ Pass 2 - Chunk {i+1:>4} | RAM: {ram_usage()}")

def finalize_full(agg_list):
    # combine every chunk's partial sums/counts/maxes into ONE true total per job
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"
        agg_spec[f"{col}_count"] = "sum"
        agg_spec[f"{col}_max"] = "max"  # max-of-maxes across chunks gives the real overall peak
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        # now actually compute the average (total sum divided by total count)
        final[f"full_{col}_avg"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
        final[f"full_{col}_peak"] = final[f"{col}_max"].round(3)
    keep = GROUP_KEYS + [f"full_{c}_avg" for c in FEATURES] + [f"full_{c}_peak" for c in FEATURES]
    return final[keep]  # only keep the columns I actually care about, drop the leftover sum/count helper columns

def finalize_early(agg_list):
    # same idea as above, just for the early-window numbers (no peak needed here, just average)
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"
        agg_spec[f"{col}_count"] = "sum"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"early_{col}"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
    keep = GROUP_KEYS + [f"early_{c}" for c in FEATURES]
    return final[keep]

full_final = finalize_full(full_chunk_aggs)
early_final = finalize_early(early_chunk_aggs)

# stitch the early-window numbers and the full-job numbers together, side by side, per job
merged = early_final.merge(full_final, on=GROUP_KEYS, how="inner")
merged = merged.merge(job_start_lookup, on=GROUP_KEYS, how="left")  # tack the job's start time back on too
merged.rename(columns={"Node": "node_id"}, inplace=True)
merged["job_start_date"] = pd.to_datetime(merged["job_start_ts"], unit="s")  # turn the raw timestamp into a readable date

print(f"\n✅ DONE! Shape: {merged.shape} | Unique jobs: {merged['id_job'].nunique():,}")

output_path = OUTPUT_DIR / OUTPUT_FILENAME
merged.to_csv(output_path, index=False)
print(f"💾 Saved → {output_path}")


# =====================================================================
# PASS 3 — same exact process, just for the 30-second window this time
# =====================================================================
EARLY_WINDOW_SEC = 30
OUTPUT_FILENAME = f"model_a_early_{EARLY_WINDOW_SEC}sec_vs_full_v4.csv"

FEATURES = [
    "power_draw_W", "utilization_gpu_pct", "utilization_memory_pct",
    "memory_used_MiB", "memory_free_MiB", "pcie_link_width_current",
    "temperature_gpu", "temperature_memory",
]

print("=" * 60)
print(f"PASS 2: extracting features for {EARLY_WINDOW_SEC}-second window...")
print("=" * 60)

early_chunk_aggs = []
full_chunk_aggs = []
pass2_cols = GROUP_KEYS + FEATURES + ["timestamp"]

for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=pass2_cols, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = clean_keys(chunk)
    for col in FEATURES:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk = chunk.merge(job_start_lookup, on=GROUP_KEYS, how="left")
    chunk["seconds_in"] = chunk["timestamp"] - chunk["job_start_ts"]

    agg_dict_full = {col: ["sum", "count", "max"] for col in FEATURES}
    full_agg = chunk.groupby(GROUP_KEYS).agg(agg_dict_full)
    full_agg.columns = ["_".join(c) for c in full_agg.columns]
    full_chunk_aggs.append(full_agg.reset_index())

    agg_dict_early = {col: ["sum", "count"] for col in FEATURES}
    early_chunk = chunk[(chunk["seconds_in"] >= 0) & (chunk["seconds_in"] <= EARLY_WINDOW_SEC)]
    if len(early_chunk):
        early_agg = early_chunk.groupby(GROUP_KEYS).agg(agg_dict_early)
        early_agg.columns = ["_".join(c) for c in early_agg.columns]
        early_chunk_aggs.append(early_agg.reset_index())

    del chunk
    if (i + 1) % 100 == 0:
        print(f"  ✅ Pass 2 - Chunk {i+1:>4} | RAM: {ram_usage()}")

def finalize_full(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"; agg_spec[f"{col}_max"] = "max"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"full_{col}_avg"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
        final[f"full_{col}_peak"] = final[f"{col}_max"].round(3)
    keep = GROUP_KEYS + [f"full_{c}_avg" for c in FEATURES] + [f"full_{c}_peak" for c in FEATURES]
    return final[keep]

def finalize_early(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"early_{col}"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
    keep = GROUP_KEYS + [f"early_{c}" for c in FEATURES]
    return final[keep]

full_final = finalize_full(full_chunk_aggs)
early_final = finalize_early(early_chunk_aggs)

merged = early_final.merge(full_final, on=GROUP_KEYS, how="inner")
merged = merged.merge(job_start_lookup, on=GROUP_KEYS, how="left")
merged.rename(columns={"Node": "node_id"}, inplace=True)
merged["job_start_date"] = pd.to_datetime(merged["job_start_ts"], unit="s")

print(f"\n✅ DONE! Shape: {merged.shape} | Unique jobs: {merged['id_job'].nunique():,}")

output_path = OUTPUT_DIR / OUTPUT_FILENAME
merged.to_csv(output_path, index=False)
print(f"💾 Saved → {output_path}")


# =====================================================================
# PASS 4 — 60-second window (rest of the passes below are all identical
# copy-paste blocks, only EARLY_WINDOW_SEC changes each time)
# =====================================================================
EARLY_WINDOW_SEC = 60
OUTPUT_FILENAME = f"model_a_early_{EARLY_WINDOW_SEC}sec_vs_full_v4.csv"

FEATURES = [
    "power_draw_W", "utilization_gpu_pct", "utilization_memory_pct",
    "memory_used_MiB", "memory_free_MiB", "pcie_link_width_current",
    "temperature_gpu", "temperature_memory",
]

print("=" * 60)
print(f"PASS 2: extracting features for {EARLY_WINDOW_SEC}-second window...")
print("=" * 60)

early_chunk_aggs = []
full_chunk_aggs = []
pass2_cols = GROUP_KEYS + FEATURES + ["timestamp"]

for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=pass2_cols, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = clean_keys(chunk)
    for col in FEATURES:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk = chunk.merge(job_start_lookup, on=GROUP_KEYS, how="left")
    chunk["seconds_in"] = chunk["timestamp"] - chunk["job_start_ts"]

    agg_dict_full = {col: ["sum", "count", "max"] for col in FEATURES}
    full_agg = chunk.groupby(GROUP_KEYS).agg(agg_dict_full)
    full_agg.columns = ["_".join(c) for c in full_agg.columns]
    full_chunk_aggs.append(full_agg.reset_index())

    agg_dict_early = {col: ["sum", "count"] for col in FEATURES}
    early_chunk = chunk[(chunk["seconds_in"] >= 0) & (chunk["seconds_in"] <= EARLY_WINDOW_SEC)]
    if len(early_chunk):
        early_agg = early_chunk.groupby(GROUP_KEYS).agg(agg_dict_early)
        early_agg.columns = ["_".join(c) for c in early_agg.columns]
        early_chunk_aggs.append(early_agg.reset_index())

    del chunk
    if (i + 1) % 100 == 0:
        print(f"  ✅ Pass 2 - Chunk {i+1:>4} | RAM: {ram_usage()}")

def finalize_full(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"; agg_spec[f"{col}_max"] = "max"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"full_{col}_avg"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
        final[f"full_{col}_peak"] = final[f"{col}_max"].round(3)
    keep = GROUP_KEYS + [f"full_{c}_avg" for c in FEATURES] + [f"full_{c}_peak" for c in FEATURES]
    return final[keep]

def finalize_early(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"early_{col}"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
    keep = GROUP_KEYS + [f"early_{c}" for c in FEATURES]
    return final[keep]

full_final = finalize_full(full_chunk_aggs)
early_final = finalize_early(early_chunk_aggs)

merged = early_final.merge(full_final, on=GROUP_KEYS, how="inner")
merged = merged.merge(job_start_lookup, on=GROUP_KEYS, how="left")
merged.rename(columns={"Node": "node_id"}, inplace=True)
merged["job_start_date"] = pd.to_datetime(merged["job_start_ts"], unit="s")

print(f"\n✅ DONE! Shape: {merged.shape} | Unique jobs: {merged['id_job'].nunique():,}")

output_path = OUTPUT_DIR / OUTPUT_FILENAME
merged.to_csv(output_path, index=False)
print(f"💾 Saved → {output_path}")


# =====================================================================
# PASS 5 — 120-second window (same block again, just the number changes)
# =====================================================================
EARLY_WINDOW_SEC = 120
OUTPUT_FILENAME = f"model_a_early_{EARLY_WINDOW_SEC}sec_vs_full_v4.csv"

FEATURES = [
    "power_draw_W", "utilization_gpu_pct", "utilization_memory_pct",
    "memory_used_MiB", "memory_free_MiB", "pcie_link_width_current",
    "temperature_gpu", "temperature_memory",
]

print("=" * 60)
print(f"PASS 2: extracting features for {EARLY_WINDOW_SEC}-second window...")
print("=" * 60)

early_chunk_aggs = []
full_chunk_aggs = []
pass2_cols = GROUP_KEYS + FEATURES + ["timestamp"]

for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=pass2_cols, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = clean_keys(chunk)
    for col in FEATURES:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk = chunk.merge(job_start_lookup, on=GROUP_KEYS, how="left")
    chunk["seconds_in"] = chunk["timestamp"] - chunk["job_start_ts"]

    agg_dict_full = {col: ["sum", "count", "max"] for col in FEATURES}
    full_agg = chunk.groupby(GROUP_KEYS).agg(agg_dict_full)
    full_agg.columns = ["_".join(c) for c in full_agg.columns]
    full_chunk_aggs.append(full_agg.reset_index())

    agg_dict_early = {col: ["sum", "count"] for col in FEATURES}
    early_chunk = chunk[(chunk["seconds_in"] >= 0) & (chunk["seconds_in"] <= EARLY_WINDOW_SEC)]
    if len(early_chunk):
        early_agg = early_chunk.groupby(GROUP_KEYS).agg(agg_dict_early)
        early_agg.columns = ["_".join(c) for c in early_agg.columns]
        early_chunk_aggs.append(early_agg.reset_index())

    del chunk
    if (i + 1) % 100 == 0:
        print(f"  ✅ Pass 2 - Chunk {i+1:>4} | RAM: {ram_usage()}")

def finalize_full(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"; agg_spec[f"{col}_max"] = "max"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"full_{col}_avg"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
        final[f"full_{col}_peak"] = final[f"{col}_max"].round(3)
    keep = GROUP_KEYS + [f"full_{c}_avg" for c in FEATURES] + [f"full_{c}_peak" for c in FEATURES]
    return final[keep]

def finalize_early(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"early_{col}"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
    keep = GROUP_KEYS + [f"early_{c}" for c in FEATURES]
    return final[keep]

full_final = finalize_full(full_chunk_aggs)
early_final = finalize_early(early_chunk_aggs)

merged = early_final.merge(full_final, on=GROUP_KEYS, how="inner")
merged = merged.merge(job_start_lookup, on=GROUP_KEYS, how="left")
merged.rename(columns={"Node": "node_id"}, inplace=True)
merged["job_start_date"] = pd.to_datetime(merged["job_start_ts"], unit="s")

print(f"\n✅ DONE! Shape: {merged.shape} | Unique jobs: {merged['id_job'].nunique():,}")

output_path = OUTPUT_DIR / OUTPUT_FILENAME
merged.to_csv(output_path, index=False)
print(f"💾 Saved → {output_path}")


# =====================================================================
# PASS 6 — 180-second window
# =====================================================================
EARLY_WINDOW_SEC = 180
OUTPUT_FILENAME = f"model_a_early_{EARLY_WINDOW_SEC}sec_vs_full_v4.csv"

FEATURES = [
    "power_draw_W", "utilization_gpu_pct", "utilization_memory_pct",
    "memory_used_MiB", "memory_free_MiB", "pcie_link_width_current",
    "temperature_gpu", "temperature_memory",
]

print("=" * 60)
print(f"PASS 2: extracting features for {EARLY_WINDOW_SEC}-second window...")
print("=" * 60)

early_chunk_aggs = []
full_chunk_aggs = []
pass2_cols = GROUP_KEYS + FEATURES + ["timestamp"]

for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=pass2_cols, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = clean_keys(chunk)
    for col in FEATURES:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk = chunk.merge(job_start_lookup, on=GROUP_KEYS, how="left")
    chunk["seconds_in"] = chunk["timestamp"] - chunk["job_start_ts"]

    agg_dict_full = {col: ["sum", "count", "max"] for col in FEATURES}
    full_agg = chunk.groupby(GROUP_KEYS).agg(agg_dict_full)
    full_agg.columns = ["_".join(c) for c in full_agg.columns]
    full_chunk_aggs.append(full_agg.reset_index())

    agg_dict_early = {col: ["sum", "count"] for col in FEATURES}
    early_chunk = chunk[(chunk["seconds_in"] >= 0) & (chunk["seconds_in"] <= EARLY_WINDOW_SEC)]
    if len(early_chunk):
        early_agg = early_chunk.groupby(GROUP_KEYS).agg(agg_dict_early)
        early_agg.columns = ["_".join(c) for c in early_agg.columns]
        early_chunk_aggs.append(early_agg.reset_index())

    del chunk
    if (i + 1) % 100 == 0:
        print(f"  ✅ Pass 2 - Chunk {i+1:>4} | RAM: {ram_usage()}")

def finalize_full(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"; agg_spec[f"{col}_max"] = "max"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"full_{col}_avg"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
        final[f"full_{col}_peak"] = final[f"{col}_max"].round(3)
    keep = GROUP_KEYS + [f"full_{c}_avg" for c in FEATURES] + [f"full_{c}_peak" for c in FEATURES]
    return final[keep]

def finalize_early(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"early_{col}"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
    keep = GROUP_KEYS + [f"early_{c}" for c in FEATURES]
    return final[keep]

full_final = finalize_full(full_chunk_aggs)
early_final = finalize_early(early_chunk_aggs)

merged = early_final.merge(full_final, on=GROUP_KEYS, how="inner")
merged = merged.merge(job_start_lookup, on=GROUP_KEYS, how="left")
merged.rename(columns={"Node": "node_id"}, inplace=True)
merged["job_start_date"] = pd.to_datetime(merged["job_start_ts"], unit="s")

print(f"\n✅ DONE! Shape: {merged.shape} | Unique jobs: {merged['id_job'].nunique():,}")

output_path = OUTPUT_DIR / OUTPUT_FILENAME
merged.to_csv(output_path, index=False)
print(f"💾 Saved → {output_path}")


# =====================================================================
# PASS 7 — 240-second window
# =====================================================================
EARLY_WINDOW_SEC = 240
OUTPUT_FILENAME = f"model_a_early_{EARLY_WINDOW_SEC}sec_vs_full_v4.csv"

FEATURES = [
    "power_draw_W", "utilization_gpu_pct", "utilization_memory_pct",
    "memory_used_MiB", "memory_free_MiB", "pcie_link_width_current",
    "temperature_gpu", "temperature_memory",
]

print("=" * 60)
print(f"PASS 2: extracting features for {EARLY_WINDOW_SEC}-second window...")
print("=" * 60)

early_chunk_aggs = []
full_chunk_aggs = []
pass2_cols = GROUP_KEYS + FEATURES + ["timestamp"]

for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=pass2_cols, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = clean_keys(chunk)
    for col in FEATURES:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk = chunk.merge(job_start_lookup, on=GROUP_KEYS, how="left")
    chunk["seconds_in"] = chunk["timestamp"] - chunk["job_start_ts"]

    agg_dict_full = {col: ["sum", "count", "max"] for col in FEATURES}
    full_agg = chunk.groupby(GROUP_KEYS).agg(agg_dict_full)
    full_agg.columns = ["_".join(c) for c in full_agg.columns]
    full_chunk_aggs.append(full_agg.reset_index())

    agg_dict_early = {col: ["sum", "count"] for col in FEATURES}
    early_chunk = chunk[(chunk["seconds_in"] >= 0) & (chunk["seconds_in"] <= EARLY_WINDOW_SEC)]
    if len(early_chunk):
        early_agg = early_chunk.groupby(GROUP_KEYS).agg(agg_dict_early)
        early_agg.columns = ["_".join(c) for c in early_agg.columns]
        early_chunk_aggs.append(early_agg.reset_index())

    del chunk
    if (i + 1) % 100 == 0:
        print(f"  ✅ Pass 2 - Chunk {i+1:>4} | RAM: {ram_usage()}")

def finalize_full(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"; agg_spec[f"{col}_max"] = "max"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"full_{col}_avg"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
        final[f"full_{col}_peak"] = final[f"{col}_max"].round(3)
    keep = GROUP_KEYS + [f"full_{c}_avg" for c in FEATURES] + [f"full_{c}_peak" for c in FEATURES]
    return final[keep]

def finalize_early(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"early_{col}"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
    keep = GROUP_KEYS + [f"early_{c}" for c in FEATURES]
    return final[keep]

full_final = finalize_full(full_chunk_aggs)
early_final = finalize_early(early_chunk_aggs)

merged = early_final.merge(full_final, on=GROUP_KEYS, how="inner")
merged = merged.merge(job_start_lookup, on=GROUP_KEYS, how="left")
merged.rename(columns={"Node": "node_id"}, inplace=True)
merged["job_start_date"] = pd.to_datetime(merged["job_start_ts"], unit="s")

print(f"\n✅ DONE! Shape: {merged.shape} | Unique jobs: {merged['id_job'].nunique():,}")

output_path = OUTPUT_DIR / OUTPUT_FILENAME
merged.to_csv(output_path, index=False)
print(f"💾 Saved → {output_path}")


# =====================================================================
# PASS 8 — 300-second window
# =====================================================================
EARLY_WINDOW_SEC = 300
OUTPUT_FILENAME = f"model_a_early_{EARLY_WINDOW_SEC}sec_vs_full_v4.csv"

FEATURES = [
    "power_draw_W", "utilization_gpu_pct", "utilization_memory_pct",
    "memory_used_MiB", "memory_free_MiB", "pcie_link_width_current",
    "temperature_gpu", "temperature_memory",
]

print("=" * 60)
print(f"PASS 2: extracting features for {EARLY_WINDOW_SEC}-second window...")
print("=" * 60)

early_chunk_aggs = []
full_chunk_aggs = []
pass2_cols = GROUP_KEYS + FEATURES + ["timestamp"]

for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=pass2_cols, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = clean_keys(chunk)
    for col in FEATURES:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk = chunk.merge(job_start_lookup, on=GROUP_KEYS, how="left")
    chunk["seconds_in"] = chunk["timestamp"] - chunk["job_start_ts"]

    agg_dict_full = {col: ["sum", "count", "max"] for col in FEATURES}
    full_agg = chunk.groupby(GROUP_KEYS).agg(agg_dict_full)
    full_agg.columns = ["_".join(c) for c in full_agg.columns]
    full_chunk_aggs.append(full_agg.reset_index())

    agg_dict_early = {col: ["sum", "count"] for col in FEATURES}
    early_chunk = chunk[(chunk["seconds_in"] >= 0) & (chunk["seconds_in"] <= EARLY_WINDOW_SEC)]
    if len(early_chunk):
        early_agg = early_chunk.groupby(GROUP_KEYS).agg(agg_dict_early)
        early_agg.columns = ["_".join(c) for c in early_agg.columns]
        early_chunk_aggs.append(early_agg.reset_index())

    del chunk
    if (i + 1) % 100 == 0:
        print(f"  ✅ Pass 2 - Chunk {i+1:>4} | RAM: {ram_usage()}")

def finalize_full(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"; agg_spec[f"{col}_max"] = "max"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"full_{col}_avg"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
        final[f"full_{col}_peak"] = final[f"{col}_max"].round(3)
    keep = GROUP_KEYS + [f"full_{c}_avg" for c in FEATURES] + [f"full_{c}_peak" for c in FEATURES]
    return final[keep]

def finalize_early(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"early_{col}"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
    keep = GROUP_KEYS + [f"early_{c}" for c in FEATURES]
    return final[keep]

full_final = finalize_full(full_chunk_aggs)
early_final = finalize_early(early_chunk_aggs)

merged = early_final.merge(full_final, on=GROUP_KEYS, how="inner")
merged = merged.merge(job_start_lookup, on=GROUP_KEYS, how="left")
merged.rename(columns={"Node": "node_id"}, inplace=True)
merged["job_start_date"] = pd.to_datetime(merged["job_start_ts"], unit="s")

print(f"\n✅ DONE! Shape: {merged.shape} | Unique jobs: {merged['id_job'].nunique():,}")

output_path = OUTPUT_DIR / OUTPUT_FILENAME
merged.to_csv(output_path, index=False)
print(f"💾 Saved → {output_path}")


# =====================================================================
# PASS 9 — 360-second window
# =====================================================================
EARLY_WINDOW_SEC = 360
OUTPUT_FILENAME = f"model_a_early_{EARLY_WINDOW_SEC}sec_vs_full_v4.csv"

FEATURES = [
    "power_draw_W", "utilization_gpu_pct", "utilization_memory_pct",
    "memory_used_MiB", "memory_free_MiB", "pcie_link_width_current",
    "temperature_gpu", "temperature_memory",
]

print("=" * 60)
print(f"PASS 2: extracting features for {EARLY_WINDOW_SEC}-second window...")
print("=" * 60)

early_chunk_aggs = []
full_chunk_aggs = []
pass2_cols = GROUP_KEYS + FEATURES + ["timestamp"]

for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=pass2_cols, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = clean_keys(chunk)
    for col in FEATURES:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk = chunk.merge(job_start_lookup, on=GROUP_KEYS, how="left")
    chunk["seconds_in"] = chunk["timestamp"] - chunk["job_start_ts"]

    agg_dict_full = {col: ["sum", "count", "max"] for col in FEATURES}
    full_agg = chunk.groupby(GROUP_KEYS).agg(agg_dict_full)
    full_agg.columns = ["_".join(c) for c in full_agg.columns]
    full_chunk_aggs.append(full_agg.reset_index())

    agg_dict_early = {col: ["sum", "count"] for col in FEATURES}
    early_chunk = chunk[(chunk["seconds_in"] >= 0) & (chunk["seconds_in"] <= EARLY_WINDOW_SEC)]
    if len(early_chunk):
        early_agg = early_chunk.groupby(GROUP_KEYS).agg(agg_dict_early)
        early_agg.columns = ["_".join(c) for c in early_agg.columns]
        early_chunk_aggs.append(early_agg.reset_index())

    del chunk
    if (i + 1) % 100 == 0:
        print(f"  ✅ Pass 2 - Chunk {i+1:>4} | RAM: {ram_usage()}")

def finalize_full(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"; agg_spec[f"{col}_max"] = "max"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"full_{col}_avg"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
        final[f"full_{col}_peak"] = final[f"{col}_max"].round(3)
    keep = GROUP_KEYS + [f"full_{c}_avg" for c in FEATURES] + [f"full_{c}_peak" for c in FEATURES]
    return final[keep]

def finalize_early(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"early_{col}"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
    keep = GROUP_KEYS + [f"early_{c}" for c in FEATURES]
    return final[keep]

full_final = finalize_full(full_chunk_aggs)
early_final = finalize_early(early_chunk_aggs)

merged = early_final.merge(full_final, on=GROUP_KEYS, how="inner")
merged = merged.merge(job_start_lookup, on=GROUP_KEYS, how="left")
merged.rename(columns={"Node": "node_id"}, inplace=True)
merged["job_start_date"] = pd.to_datetime(merged["job_start_ts"], unit="s")

print(f"\n✅ DONE! Shape: {merged.shape} | Unique jobs: {merged['id_job'].nunique():,}")

output_path = OUTPUT_DIR / OUTPUT_FILENAME
merged.to_csv(output_path, index=False)
print(f"💾 Saved → {output_path}")


# =====================================================================
# PASS 10 — 600-second window (the last one, this is my "ceiling" model)
# =====================================================================
EARLY_WINDOW_SEC = 600
OUTPUT_FILENAME = f"model_a_early_{EARLY_WINDOW_SEC}sec_vs_full_v4.csv"

FEATURES = [
    "power_draw_W", "utilization_gpu_pct", "utilization_memory_pct",
    "memory_used_MiB", "memory_free_MiB", "pcie_link_width_current",
    "temperature_gpu", "temperature_memory",
]

print("=" * 60)
print(f"PASS 2: extracting features for {EARLY_WINDOW_SEC}-second window...")
print("=" * 60)

early_chunk_aggs = []
full_chunk_aggs = []
pass2_cols = GROUP_KEYS + FEATURES + ["timestamp"]

for i, chunk in enumerate(pd.read_csv(RAW_FILE, usecols=pass2_cols, chunksize=CHUNKSIZE, low_memory=False)):
    chunk = clean_keys(chunk)
    for col in FEATURES:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    chunk = chunk.merge(job_start_lookup, on=GROUP_KEYS, how="left")
    chunk["seconds_in"] = chunk["timestamp"] - chunk["job_start_ts"]

    agg_dict_full = {col: ["sum", "count", "max"] for col in FEATURES}
    full_agg = chunk.groupby(GROUP_KEYS).agg(agg_dict_full)
    full_agg.columns = ["_".join(c) for c in full_agg.columns]
    full_chunk_aggs.append(full_agg.reset_index())

    agg_dict_early = {col: ["sum", "count"] for col in FEATURES}
    early_chunk = chunk[(chunk["seconds_in"] >= 0) & (chunk["seconds_in"] <= EARLY_WINDOW_SEC)]
    if len(early_chunk):
        early_agg = early_chunk.groupby(GROUP_KEYS).agg(agg_dict_early)
        early_agg.columns = ["_".join(c) for c in early_agg.columns]
        early_chunk_aggs.append(early_agg.reset_index())

    del chunk
    if (i + 1) % 100 == 0:
        print(f"  ✅ Pass 2 - Chunk {i+1:>4} | RAM: {ram_usage()}")

def finalize_full(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"; agg_spec[f"{col}_max"] = "max"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"full_{col}_avg"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
        final[f"full_{col}_peak"] = final[f"{col}_max"].round(3)
    keep = GROUP_KEYS + [f"full_{c}_avg" for c in FEATURES] + [f"full_{c}_peak" for c in FEATURES]
    return final[keep]

def finalize_early(agg_list):
    combined = pd.concat(agg_list, ignore_index=True)
    agg_spec = {}
    for col in FEATURES:
        agg_spec[f"{col}_sum"] = "sum"; agg_spec[f"{col}_count"] = "sum"
    final = combined.groupby(GROUP_KEYS, as_index=False).agg(agg_spec)
    for col in FEATURES:
        final[f"early_{col}"] = (final[f"{col}_sum"] / final[f"{col}_count"]).round(3)
    keep = GROUP_KEYS + [f"early_{c}" for c in FEATURES]
    return final[keep]

full_final = finalize_full(full_chunk_aggs)
early_final = finalize_early(early_chunk_aggs)

merged = early_final.merge(full_final, on=GROUP_KEYS, how="inner")
merged = merged.merge(job_start_lookup, on=GROUP_KEYS, how="left")
merged.rename(columns={"Node": "node_id"}, inplace=True)
merged["job_start_date"] = pd.to_datetime(merged["job_start_ts"], unit="s")

print(f"\n✅ DONE! Shape: {merged.shape} | Unique jobs: {merged['id_job'].nunique():,}")

output_path = OUTPUT_DIR / OUTPUT_FILENAME
merged.to_csv(output_path, index=False)
print(f"💾 Saved → {output_path}")